# Lab6.2 — Hardware Streaming Pipeline: Design and Dataflow

**Core question:** what changes architecturally when Resize and Sobel are fused with AXI4-Stream instead of storing an intermediate image in DDR?

This is the **design-heavy** notebook of Lab6. We will implement a functional **three-line Sobel streaming model** on ARM, investigate AXI4-Stream handshakes and quantify *theoretical* DDR payloads. These are models; they are **not FPGA measurements**.

**Hardware target:** DDR → DMA MM2S → Resize (PL) → grayscale (PL) → Sobel (PL) → DMA S2MM → DDR. A new, experimentally verified bit/hwh overlay is required before this full path can execute in programmable logic.

## 1. Draw the architecture before running code

**Two independent accelerators:** input DDR → DMA → Resize → DMA → intermediate DDR → DMA → Sobel → DMA → output DDR.

**Fused AXI stream:** input DDR → DMA → Resize → grayscale → Sobel → DMA → output DDR.

The fused architecture removes the external intermediate frame transfer. It does **not** mean the Sobel core uses no memory: a 3×3 neighborhood typically requires line buffers (e.g. BRAM) and register windows.

**Student assignment:** annotate both diagrams with the bit width, frame shape and where backpressure may occur. Explain why a full-frame DDR buffer differs from a 3-row line buffer.

## 2. Functional streaming model: three rows, not a whole intermediate frame

Use the same image and geometry as Lab6.1. Implement Sobel at a row-oriented level by retaining the previous, current and next row. For this exercise use **zero-padding** at image boundaries, and compare against OpenCV configured with exactly the same border policy. The NumPy model below is deliberately **not** a cycle-accurate RTL simulation.

The output array is stored solely to visualize and validate results; it is not a simulated PL DDR write.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from lab6_common import load_test_image, sobel_software, compare_images

bgr = load_test_image()
TARGET = (320, 240)
small = cv2.resize(bgr, TARGET, interpolation=cv2.INTER_LINEAR)
gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

def sobel_three_rows(gray_image):
    """Functional Sobel with 3 row buffers, BORDER_CONSTANT=0."""
    if gray_image.ndim != 2 or gray_image.dtype != np.uint8:
        raise TypeError("Expected one uint8 grayscale frame.")
    h, w = gray_image.shape
    zero = np.zeros(w, dtype=np.int32)
    previous = zero
    current = gray_image[0].astype(np.int32)
    following = gray_image[1].astype(np.int32) if h > 1 else zero
    output = np.empty((h, w), dtype=np.uint8)
    for y in range(h):
        p = np.pad(previous, (1, 1), mode="constant")
        q = np.pad(current, (1, 1), mode="constant")
        n = np.pad(following, (1, 1), mode="constant")
        gx = (-p[:-2] + p[2:] - 2*q[:-2] + 2*q[2:]
              -n[:-2] + n[2:])
        gy = (-p[:-2] - 2*p[1:-1] - p[2:]
              +n[:-2] + 2*n[1:-1] + n[2:])
        magnitude = np.sqrt(gx.astype(np.float32)**2 +
                            gy.astype(np.float32)**2)
        output[y] = np.clip(np.rint(magnitude), 0, 255).astype(np.uint8)
        previous, current = current, following
        following = (gray_image[y+2].astype(np.int32)
                     if y+2 < h else zero)
    return output

reference = sobel_software(gray, border_type=cv2.BORDER_CONSTANT)
stream_model = sobel_three_rows(gray)
print("OpenCV vs 3-line model:", compare_images(reference, stream_model))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, pic, title in zip(axes,
        [reference, stream_model,
         np.abs(reference.astype(np.int16)-stream_model.astype(np.int16))],
        ["OpenCV reference", "Three-line model", "Absolute difference"]):
    ax.imshow(pic, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

### Validation questions

- How many complete rows does the algorithm need for a 3×3 window? Are all of these necessarily copied into separate BRAM blocks?
- What is the effect of the chosen boundary policy on edge pixels?
- Why can you have a small magnitude difference after identical convolution coefficients (rounding)?
- If the image width doubles, how does required line storage change? Compare this with storing an entire frame.

**Optional investigation:** change the border policy to constant vs reflected. Ensure the software reference uses the *same* policy as the model before comparing.

## 3. Understand TVALID / TREADY and frame boundaries

In AXI4-Stream, a beat is accepted **only** when TVALID and TREADY are both high at the active clock edge. The sender must hold its payload and sidebands stable while valid data is stalled. Backpressure can propagate upstream through a chain of stream IP blocks.

Below we simulate one packet with a deliberately stalling receiver. For this simple packet, TLAST marks its final beat. **Do not generalize that rule to AXI4-Stream Video:** video interfaces commonly use TUSER for start-of-frame and TLAST for end-of-line. Check the actual Resize IP interface and downstream DMA framing before implementing RTL.

In [ ]:
def simulate_stream_packet(n_pixels, stall_period=5):
    queued = list(range(n_pixels))
    accepted, trace = [], []
    pending = None
    clock = 0
    while len(accepted) < n_pixels:
        if pending is None:
            ix = len(accepted)
            pending = (queued[ix], ix == n_pixels-1)
        valid = True
        ready = clock % stall_period != stall_period-1
        data, last = pending
        transferred = bool(valid and ready)
        if len(trace) < 18:
            trace.append((clock, int(valid), int(ready),
                          data, int(last), int(transferred)))
        if transferred:
            accepted.append(data)
            pending = None
        clock += 1
    assert accepted == queued
    return clock, trace

cycles, trace = simulate_stream_packet(32, stall_period=5)
print("Clock VALID READY DATA LAST ACCEPTED")
for row in trace:
    print("%5d %5d %5d %4d %4d %8d" % row)
print("32 beats delivered in", cycles, "cycles; average:",
      round(32/cycles, 3), "beats/clock")

## 4. Estimate the eliminated **DDR payload**, not actual bandwidth

Assumptions: source image is 3-channel uint8, Resize produces a 3-channel uint8 intermediate, and grayscale Sobel produces a one-channel uint8 final frame. Ignore caches, AXI burst overhead, alignment, CPU copies and simultaneous video acquisition.

Sequential PL kernels: input bytes + **2 × intermediate bytes** (write + read) + output bytes.

Fused stream: input bytes + output bytes.

The difference is exactly two intermediate-frame payloads under this model. Actual memory-controller transactions may be higher, and the hardware may contain additional buffering or stream-width conversion.

In [ ]:
source_h, source_w, _ = bgr.shape
sizes = [(160, 120), (320, 240), (480, 320)]
print("%-12s %15s %15s %13s" %
      ("Output", "Separate MiB", "Streaming MiB", "Saving [%]"))
traffic = []
for w, h in sizes:
    source_bytes = source_w * source_h * 3
    intermediate_bytes = w * h * 3
    result_bytes = w * h  # grayscale Sobel result
    separate = source_bytes + 2*intermediate_bytes + result_bytes
    fused = source_bytes + result_bytes
    saving_percent = 100*(separate-fused)/separate
    traffic.append((w, h, separate, fused))
    print("%-12s %15.3f %15.3f %13.2f" %
          ("%dx%d" % (w, h), separate/2**20,
           fused/2**20, saving_percent))

labels = ["%dx%d" % (w, h) for w, h, _, _ in traffic]
x = np.arange(len(labels))
plt.figure(figsize=(8, 4))
plt.bar(x-0.18, [a/2**20 for _, _, a, _ in traffic],
        width=0.36, label="Separate DDR")
plt.bar(x+0.18, [b/2**20 for _, _, _, b in traffic],
        width=0.36, label="Fused AXI stream")
plt.xticks(x, labels)
plt.ylabel("Minimum DDR payload per frame [MiB]")
plt.title("Analytical model — not measured traffic")
plt.legend(); plt.tight_layout(); plt.show()

## 5. From the model to a real PYNQ-Z2 overlay

Design a block diagram in Vivado 2022.2 with these functional blocks:

1. A PS-accessible DDR connection and AXI DMA (MM2S + S2MM), with an interrupt or polling strategy.
2. A Resize IP with AXI4-Stream ports and AXI-Lite controls.
3. Pixel-format / grayscale conversion, then Sobel 3×3 with line buffers and a defined boundary policy.
4. A direct streaming link between the functional blocks, with matching TDATA, TKEEP, TVALID, TREADY, and packet/framing signals.
5. Output stream format adapted to the DMA, and a matching bit/hwh pair.

**Important integration checks:** the PYNQ-HelloWorld resizer from Lab5 is not automatically compatible with a second 8-bit Sobel IP. RGB/BGR ordering, bytes per beat, TLAST semantics, output packet length and flush/invalidate policy need to be verified. Do not load an unrelated .hwh file.

**State of hardware:** no verified fused Resize→Sobel bitstream is included in this course yet. A software line-buffer model is not evidence that the design has run in PL.

## 6. Integrated student work / technical report

Produce one 1–2 page engineering note, not five disconnected answers:

- Draw the sequential and fused architectures, marking DDR, AXI DMA, AXI-Lite and AXI4-Stream.
- Present your three-line functional validation including border policy and maximum numerical difference.
- Explain a real backpressure stall from the handshake trace.
- Calculate the DDR payload saved at an additional chosen resolution; justify each term.
- Describe a **hardware validation plan**: overlay metadata inspection, full-frame correctness, framing faults, and DMA + PL vs application end-to-end timing.

**Bridge to Lab6.3:** all reductions in DDR payload are potential architectural benefits. Whether they increase camera FPS requires measured end-to-end video behavior.